# 1 · Dataset exploration

Measures the corpus and writes `docs/dataset-report.md` with its figures.
Everything here is evidence for a modelling decision, not description for its own sake.

**Person A deliverable.**


## Setup

Clone the repository and install. The data layer needs nothing beyond the standard
library, so this is only for the model code.


In [ ]:
!git clone -q https://github.com/ManasDasri/NNDL.git
%cd NNDL
!pip install -q -e . 'matplotlib>=3.8'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


### Prepare the corpus

Extracts answer spans from `CUAD_v1.json` and assigns document-level splits.
Chunking happens at training time, because each model needs a different window.


In [ ]:
!legal-risk-prepare --cuad_json data/CUAD_v1.json --out_dir data/processed


### Measure it


In [ ]:
!legal-risk-analyze --window 300 --overlap 100


### The three findings that matter

Read these off the report rather than trusting the summary below.


In [ ]:
import json
stats = json.load(open('docs/dataset-stats.json'))
lengths, chunks = stats['document_lengths'], stats['chunk_labels']

print(f"contracts: {lengths['documents']}")
print(f"median words: {lengths['words']['median']:,.0f}   longest: {lengths['words']['max']:,.0f}")
print(f"over BERT 512:       {lengths['exceeding_bert_512']} ({lengths['exceeding_bert_512']/lengths['documents']:.1%})")
print(f"over Longformer 4096:{lengths['exceeding_longformer_4096']} ({lengths['exceeding_longformer_4096']/lengths['documents']:.1%})")
print(f"\nwindows: {chunks['chunks']:,}   carrying no label: {chunks['unlabelled_rate']:.1%}")
for name, row in chunks['per_label'].items():
    print(f"  {name:32s} {row['positive_rate']:6.2%}  pos_weight {row['pos_weight']:5.0f}x")


### Figures


In [ ]:
from IPython.display import Image, display
for name in ('document_lengths', 'chunk_positive_rates', 'label_cooccurrence', 'split_balance'):
    display(Image(f'docs/figures/{name}.png'))


### What this forces

- **97% of contracts exceed BERT's window and 70% exceed Longformer's.** Chunking is
  required for both models. Long context changes how much a window holds, not whether
  chunking is needed.
- **85% of windows carry no label.** Accuracy is meaningless; an all-negative model
  scores above 95% on every label. Report per-class precision, recall and F1.
- **Labels co-occur.** Six independent sigmoids, never a softmax.
